In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_loader import PairData

data   = PairData('../../data/real_pair_XOM_CVX.csv')
prices = data.s          # shape (2, N)
labels = data.labels     # ['XOM', 'CVX']
N      = prices.shape[1]

# timestamps → dates
df_raw = pd.read_csv('../../data/real_pair_XOM_CVX.csv', index_col=0, parse_dates=True)
df_raw.index = pd.to_datetime(df_raw.index, unit='s')
dates  = df_raw.index

train_end = 3 * N // 4

print(f'Loaded {N} days  ({dates[0].date()} → {dates[-1].date()})')
print(f'Train: {dates[0].date()} – {dates[train_end-1].date()}  ({train_end} days)')
print(f'Test:  {dates[train_end].date()} – {dates[-1].date()}  ({N - train_end} days)')

In [ ]:
log_xom = np.log(prices[0])
log_cvx = np.log(prices[1])

def sharpe(W, rf=0.0):
    r = np.diff(W) / W[:-1]
    return float((r.mean() - rf) / (r.std() + 1e-12) * np.sqrt(252))

def ann_ret(W):
    return float((W[-1] / W[0]) ** (252 / len(W)) - 1)

def rolling_z(spread, window=60):
    s = pd.Series(spread)
    return (s - s.rolling(window).mean()) / s.rolling(window).std().clip(lower=1e-8)


In [ ]:
# ── 6-month rolling beta via minimum half-life ────────────────────────────
# Every 126 days: fit beta on past 252 days that minimises AR(1) half-life.
# Half-life of AR(1)  s[t] = a + phi*s[t-1] + e  is  -log(2)/log(|phi|).

REFIT_FREQ  = 126   # ~6 months
HL_LOOKBACK = 252   # 1 year
BETA_GRID   = np.arange(0.1, 3.01, 0.05)

def min_hl_beta(log_A_win, log_B_win, grid):
    """Return (beta, half_life) minimising AR(1) HL over the window."""
    best_beta, best_hl = None, np.inf
    for b in grid:
        sp   = log_A_win - b * log_B_win
        y    = sp[1:]; X = np.column_stack([np.ones(len(y)), sp[:-1]])
        c, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        phi  = float(c[1])
        if phi <= 0 or phi >= 1:
            continue
        hl = -np.log(2) / np.log(phi)
        if hl < best_hl:
            best_hl, best_beta = hl, b
    return best_beta, best_hl

# build piecewise-constant beta and intercept arrays
beta_hl = np.full(N, np.nan)
icp_hl  = np.full(N, np.nan)
refit_t    = []   # time indices where we refit
refit_beta = []
refit_hl   = []

t = HL_LOOKBACK
while t < N:
    A_w = log_xom[t - HL_LOOKBACK : t]
    B_w = log_cvx[t - HL_LOOKBACK : t]
    b, hl = min_hl_beta(A_w, B_w, BETA_GRID)
    if b is None:
        t += REFIT_FREQ; continue
    icp = float((A_w - b * B_w).mean())
    end = min(t + REFIT_FREQ, N)
    beta_hl[t:end] = b
    icp_hl[t:end]  = icp
    refit_t.append(t)
    refit_beta.append(b)
    refit_hl.append(hl)
    t += REFIT_FREQ

spread_hl = log_xom - beta_hl * log_cvx - icp_hl

print(f'Refit points: {len(refit_t)}')
print(f'Beta range:   {min(refit_beta):.3f} – {max(refit_beta):.3f}')
print(f'HL range:     {min(refit_hl):.1f} – {max(refit_hl):.1f} days')

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle('6-month rolling min-HL beta  (lookback=1yr)', fontsize=12)

ax = axes[0]
ax.step(dates[refit_t], refit_beta, where='post', color='steelblue', lw=1.5, label='min-HL β')
ax.axhline(beta_ols, color='black', lw=0.8, ls='--', label=f'static OLS β={beta_ols:.3f}')
for rt in refit_t:
    ax.axvline(dates[rt], color='steelblue', lw=0.5, alpha=0.3)
ax.axvline(dates[train_end], color='red', lw=1.2, ls='--', alpha=0.7, label='train/test split')
ax.set_ylabel('β'); ax.legend(fontsize=9)

ax = axes[1]
ax.plot(dates, spread_hl, color='purple', lw=0.8)
ax.axhline(0, color='black', lw=0.5, ls='--')
for rt in refit_t:
    ax.axvline(dates[rt], color='steelblue', lw=0.5, alpha=0.3)
ax.axvline(dates[train_end], color='red', lw=1.2, ls='--', alpha=0.7)
ax.fill_between(dates, spread_hl, 0, where=(~np.isnan(spread_hl)) & (spread_hl > 0), alpha=0.15, color='red')
ax.fill_between(dates, spread_hl, 0, where=(~np.isnan(spread_hl)) & (spread_hl < 0), alpha=0.15, color='green')
ax.set_ylabel('Spread')

z_hl = rolling_z(spread_hl)
ax = axes[2]
ax.plot(dates, z_hl, color='purple', lw=0.7)
ax.axhline( 2, color='gray', lw=0.6, ls=':')
ax.axhline(-2, color='gray', lw=0.6, ls=':')
ax.axhline( 0, color='black', lw=0.5, ls='--')
for rt in refit_t:
    ax.axvline(dates[rt], color='steelblue', lw=0.5, alpha=0.3)
ax.axvline(dates[train_end], color='red', lw=1.2, ls='--', alpha=0.7, label='train/test split')
ax.fill_between(dates, z_hl,  2, where=(~np.isnan(z_hl)) & (z_hl >  2), alpha=0.2, color='red')
ax.fill_between(dates, z_hl, -2, where=(~np.isnan(z_hl)) & (z_hl < -2), alpha=0.2, color='green')
ax.set_ylabel('Z-score'); ax.set_xlabel('Date'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# ── RL on min-HL spread — rolling z, beta locked at entry only ────────────
import sys; sys.path.insert(0, '..')
from ddqn_v2 import DoubleDQN

# Precompute rolling z-scores using piecewise-beta spread (used when flat)
_WINS    = (10, 45, 100)
_z_multi = np.column_stack([rolling_z(spread_hl, w).values for w in _WINS])  # (N, 3)

class SpreadRLEnv:
    """
    Beta is locked at entry and released on exit.
    Z-scores always roll forward (no fixed reference) using the locked spread.
    """
    _W = (10, 45, 100)

    def __init__(self, prices, log_A, log_B, beta_arr, icp_arr,
                 z_multi, spread_arr,
                 initial_wealth=10_000, tc=0.0001, episode_len=252):
        self.prices      = prices.astype(np.float64)
        self.log_A       = log_A;       self.log_B      = log_B
        self.beta_arr    = beta_arr;    self.icp_arr    = icp_arr
        self.z_multi     = z_multi;     self.spread_arr = spread_arr
        self.W0          = float(initial_wealth)
        self.tc          = float(tc)
        self.episode_len = int(episode_len)
        self.N           = prices.shape[1]

    def _safe_t(self): return min(self._t, self.N - 1)

    def _wealth(self):
        return float(self._cash + self._shares @ self.prices[:, self._safe_t()])

    def _locked_spread(self, t):
        return float(self.log_A[t] - self._e_beta * self.log_B[t] - self._e_icp)

    def _buf_z(self, i):
        arr = np.array(self._bufs[i])
        if len(arr) < 2:
            return 0.0
        return float((arr[-1] - arr.mean()) / (arr.std() + 1e-8))

    def _obs(self):
        t = self._safe_t()
        if self._pos == 0:
            sp = float(self.spread_arr[t]) if not np.isnan(self.spread_arr[t]) else 0.
            z  = np.where(np.isnan(self.z_multi[t]), 0., self.z_multi[t]).astype(np.float32)
        else:
            sp = self._locked_spread(t)
            z  = np.array([self._buf_z(i) for i in range(3)], dtype=np.float32)
        return np.array([sp, z[0], z[1], z[2], float(self._pos)], dtype=np.float32)

    def _enter(self, direction, beta):
        t = self._safe_t()
        p0, p1 = self.prices[0, t], self.prices[1, t]
        C = self._wealth()
        d0, d1 = C / (1 + beta), beta * C / (1 + beta)
        self._shares[0] =  direction * d0 / p0
        self._shares[1] = -direction * d1 / p1
        self._cash     += direction * (d1 - d0) - self.tc * (d0 + d1)
        self._pos = direction; self._e_beta = beta; self._e_icp = self.icp_arr[t]
        # seed rolling buffers with last w days of locked spread
        for i, w in enumerate(self._W):
            t0 = max(0, t - w + 1)
            hist = [self._locked_spread(s) for s in range(t0, t + 1)]
            self._bufs[i] = hist

    def _exit(self):
        t = self._safe_t()
        p0, p1 = self.prices[0, t], self.prices[1, t]
        notional = abs(self._shares[0])*p0 + abs(self._shares[1])*p1
        self._cash = self._wealth() - self.tc * notional
        self._shares[:] = 0; self._pos = 0; self._e_beta = self._e_icp = None
        self._bufs = [[] for _ in self._W]

    def reset(self, t_start=0):
        self._t      = int(t_start)
        self._end    = self._t + self.episode_len
        self._cash   = self.W0
        self._shares = np.zeros(2); self._pos = 0
        self._e_beta = self._e_icp = None
        self._bufs   = [[] for _ in self._W]
        return self._obs()

    def step(self, action):
        desired = {0: 0, 1: 1, 2: -1}[int(action)]

        # stop-loss: force exit if z10 moves > 3 against position
        if self._pos != 0 and len(self._bufs[0]) >= 2:
            z10 = self._buf_z(0)
            if (self._pos == 1 and z10 < -3) or (self._pos == -1 and z10 > 3):
                desired = 0

        old_w = self._wealth()
        if desired != self._pos:
            if self._pos != 0: self._exit()
            if desired   != 0: self._enter(desired, float(self.beta_arr[self._safe_t()]))

        self._t += 1

        # roll buffers forward with new spread (after time advance)
        if self._pos != 0:
            t = self._safe_t()
            sp = self._locked_spread(t)
            for i, w in enumerate(self._W):
                self._bufs[i].append(sp)
                if len(self._bufs[i]) > w:
                    self._bufs[i] = self._bufs[i][-w:]

        done = bool(self._t >= self._end)
        return self._obs(), float(self._wealth() - old_w), done, False, {}


# ── Training ───────────────────────────────────────────────────────────────
SEED            = 42
N_EPISODES      = 800
EPISODE_LEN     = 252
TRAIN_FREQ      = 4
LOG_EVERY       = 100
T0_MIN          = int(HL_LOOKBACK + max(_WINS) + 1)
T0_MAX          = int(train_end - EPISODE_LEN - 1)

rng     = np.random.default_rng(SEED)
env_rl  = SpreadRLEnv(prices, log_xom, log_cvx, beta_hl, icp_hl,
                      _z_multi, spread_hl, episode_len=EPISODE_LEN)
agent   = DoubleDQN(
    state_dim=5, action_dim=3, hidden=64, lr=3e-4,
    gamma=0.99, tau=0.005, buffer_size=int(1e5), batch_size=256,
    eps_start=1.0, eps_end=0.05,
    eps_decay_steps=N_EPISODES * EPISODE_LEN // 2, grad_clip=1.0,
)

ep_rewards = []; ep_wealth = []; total_steps = 0

for ep in range(N_EPISODES):
    t0  = int(rng.integers(T0_MIN, T0_MAX + 1))
    obs = env_rl.reset(t_start=t0)
    done = False; ep_r = 0.
    while not done:
        action = agent.select_action(obs)
        obs2, r, done, _, _ = env_rl.step(action)
        agent.replay_buffer.add(obs.copy(), action, r, obs2.copy(), done)
        obs = obs2; ep_r += r; total_steps += 1
        if total_steps % TRAIN_FREQ == 0:
            agent.train()
    ep_rewards.append(ep_r); ep_wealth.append(env_rl._wealth())
    if (ep + 1) % LOG_EVERY == 0:
        print(f'ep {ep+1:4d}  eps={agent.epsilon:.3f}  '
              f'reward={np.mean(ep_rewards[-LOG_EVERY:]):+8.1f}  '
              f'wealth={np.mean(ep_wealth[-LOG_EVERY:]):.0f}')

print('Training done.')


# ── Evaluate ───────────────────────────────────────────────────────────────
def eval_agent(t_start, t_end):
    env = SpreadRLEnv(prices, log_xom, log_cvx, beta_hl, icp_hl,
                      _z_multi, spread_hl, episode_len=t_end - t_start)
    obs = env.reset(t_start=t_start)
    wealth = [env._wealth()]; pos_hist = []; sp_hist = []
    done = False
    while not done:
        t = env._safe_t()
        sp_hist.append(float(spread_hl[t]) if env._pos == 0 and not np.isnan(spread_hl[t])
                       else (env._locked_spread(t) if env._pos != 0 else 0.))
        act = agent.select_action(obs, deterministic=True)
        obs, _, done, _, _ = env.step(act)
        pos_hist.append(env._pos); wealth.append(env._wealth())
    if env._pos != 0:
        env._exit(); wealth[-1] = env._wealth()
    return np.array(wealth), np.array(pos_hist), np.array(sp_hist)

for label, t0, t1 in [('Train', T0_MIN, train_end), ('Test', train_end, N)]:
    W, pos, sp = eval_agent(t0, t1)
    T = len(pos); t_x = np.arange(T)
    prev    = np.concatenate([[0], pos[:-1]])
    buy_i   = np.where((pos ==  1) & (prev !=  1))[0]
    sh_i    = np.where((pos == -1) & (prev != -1))[0]
    ex_i    = np.where((pos ==  0) & (prev !=  0))[0]
    n_trades = int((np.diff(np.concatenate([[0], pos])) != 0).sum())
    print(f'{label}: long={int((pos==1).sum())}d  short={int((pos==-1).sum())}d  '
          f'flat={int((pos==0).sum())}d  trades={n_trades}  '
          f'SR={sharpe(W):.3f}  ann={ann_ret(W)*100:+.2f}%')

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    fig.suptitle(f'DDQN v3 — {label}  SR={sharpe(W):.3f}', fontsize=11)
    ax = axes[0]
    ax.plot(t_x, sp, color='black', lw=0.6)
    ax.axhline(0, color='gray', lw=0.7, ls='--')
    for rt in refit_t:
        rt_rel = rt - t0
        if 0 <= rt_rel < T:
            ax.axvline(rt_rel, color='steelblue', lw=0.6, alpha=0.4, ls=':')
    ax.scatter(buy_i, sp[buy_i], color='green', s=40, zorder=3, label='Buy')
    ax.scatter(sh_i,  sp[sh_i],  color='red',   s=40, zorder=3, label='Short')
    ax.scatter(ex_i,  sp[ex_i],  color='blue',  s=40, zorder=3, label='Exit')
    ax.set_ylabel('Spread'); ax.legend(fontsize=9)
    ax = axes[1]
    t_w = np.arange(T + 1)
    ax.plot(t_w, W - W[0], color='navy', lw=1.2)
    ax.axhline(0, color='black', lw=0.6, ls='--')
    ax.fill_between(t_w, W-W[0], 0, where=(W>=W[0]), alpha=0.15, color='green')
    ax.fill_between(t_w, W-W[0], 0, where=(W< W[0]), alpha=0.15, color='red')
    ax.set_ylabel('Cumul PnL ($)'); ax.set_xlabel('Days')
    plt.tight_layout(); plt.show()
